# 3. Fine-tuning BERT for News Classification

We fine-tune `bert-base-uncased` (Devlin et al., 2019) with a linear classification head on the AG News training split. Training runs for three epochs with AdamW (learning rate 2e-5, weight decay 0.01, 6% linear warm-up), batch size 32, maximum sequence length 128 and mixed precision. The checkpoint with the highest validation accuracy is restored and evaluated once on the official test split.

This notebook is self-contained and runs on Google Colab with a GPU runtime (Runtime > Change runtime type > T4 GPU).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EmpireNetwork123/explainable-news-classification/blob/main/notebooks/03_bert_finetune.ipynb)

## 3.1 Environment

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/explainable-news-classification").exists():
        !git clone -q https://github.com/EmpireNetwork123/explainable-news-classification.git /content/explainable-news-classification
    %cd /content/explainable-news-classification
    !pip install -q -r requirements.txt
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(ROOT)
sys.path.insert(0, os.getcwd())

In [ ]:
import torch, transformers

print("PyTorch", torch.__version__, "| Transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

## 3.2 Data

In [ ]:
from src import config
from src.download_data import download
from src.preprocess import preprocess_and_save

if not (config.RAW_DIR / "ag_news_train.csv").exists():
    download()
preprocess_and_save()

## 3.3 Fine-tuning

In [ ]:
!python -m src.train_bert

## 3.4 Test-set evaluation

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

metrics = json.loads((config.METRICS_DIR / "bert_metrics.json").read_text())
report = pd.DataFrame(metrics["per_class"]).T.loc[config.LABEL_NAMES + ["macro avg"], ["precision", "recall", "f1-score", "support"]]
print(f"Accuracy: {metrics['accuracy']:.4f}   Macro-F1: {metrics['macro_f1']:.4f}   ({metrics['device']})")
report.round(4)

In [ ]:
log = json.loads((config.METRICS_DIR / "bert_log_history.json").read_text())
pd.DataFrame([e for e in log if "eval_loss" in e])[["epoch", "eval_loss", "eval_accuracy"]].round(4)

In [ ]:
display(Image(filename=str(config.FIGURES_DIR / "bert_training_curve.png"), width=460))
display(Image(filename=str(config.FIGURES_DIR / "bert_confusion_matrix.png"), width=480))

## 3.5 Explaining BERT predictions with LIME

We explain the same test article as in the Naive Bayes notebook, which allows a direct comparison of the evidence each model relies on.

In [ ]:
from src.explain_model import explain

exp = explain(model="bert", index=0)
display(Image(filename=str(config.FIGURES_DIR / "lime_bert_test0.png"), width=640))

The interactive version is written to `results/figures/lime_bert_test0.html`.

## 3.6 Export results

On Colab, the metrics, figures and this executed notebook are packaged into `bert_results.zip` for download.

In [ ]:
if IN_COLAB:
    import json as _json
    import shutil
    from google.colab import _message, files

    export = Path("/content/bert_results")
    shutil.rmtree(export, ignore_errors=True)
    (export / "metrics").mkdir(parents=True)
    (export / "figures").mkdir(parents=True)
    for f in config.METRICS_DIR.glob("bert_*"):
        shutil.copy(f, export / "metrics")
    for f in list(config.FIGURES_DIR.glob("bert_*")) + list(config.FIGURES_DIR.glob("lime_bert_*")):
        shutil.copy(f, export / "figures")
    notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    (export / "03_bert_finetune.ipynb").write_text(_json.dumps(notebook))
    shutil.make_archive("/content/bert_results", "zip", export)
    files.download("/content/bert_results.zip")